In [1]:
import pandas as pd
# import numpy as np
# from sklearn.utils import shuffle

# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns
# import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_curve, auc

# from sklearn.preprocessing import label_binarize
# from sklearn.metrics import roc_curve, auc
# from sklearn.utils import shuffle

# from sklearn.multiclass import OneVsRestClassifier
# from sklearn import svm, datasets

# from tensorflow.keras import regularizers

# from keras_tuner import RandomSearch
# import keras_tuner as kt

import json
import copy
# import gc
# import random

2025-11-07 00:27:49.815330: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-07 00:27:49.985868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-07 00:27:50.889595: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# !conda install pandas

In [3]:
print(tf.__version__)

2.20.0


In [4]:
# from IPython.core.display import display, HTML
# display(HTML("<style>.container { width:99% !important; }</style>"))

In [5]:
# Dc=pd.read_csv('./data/cg_train_2019_to_2020-09_119124.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2021-09_171515.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2022-09_228433.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2023-09_273708.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2025-09_384615.csv')

# data=pd.read_csv("data/time_fold/cg_train_2019_to_2020-09_times_kfold119124_"+str(k)+".csv",index_col=0) 
# data=pd.read_csv("data/time_fold/cg_train_2019_to_2021-09_times_kfold171515_"+str(k)+".csv",index_col=0)
# data=pd.read_csv("data/time_fold/cg_train_2019_to_2022-09_times_kfold228433_"+str(k)+".csv",index_col=0) 

# data=pd.read_csv("data/time_fold/cg_train_2019_to_2023-09_times_kfold273708_"+str(k)+".csv",index_col=0)
# data=pd.read_csv("data/time_fold/cg_train_2019_to_2024-09_times_kfold329692_"+str(k)+".csv",index_col=0) 
# data=pd.read_csv("data/time_fold/cg_train_2019_to_2025-09_times_kfold384615_"+str(k)+".csv",index_col=0)

In [18]:
# Dc=pd.read_csv('./data/cg_338377.csv')
# Dp=pd.read_csv('./data/py_32319.csv')

In [19]:
lst=[['male', 'female','zy','mz','age','mth','wk','wbc','neu','lym','mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct', 'plt', 'pdw', 'pct', 'plcr', 'label'],
    ['male', 'female','zy', 'mz', 'age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'hct','plt','label']]
fnams=['all','tim_plt3','tim_plt3_sex_zy_mz','tim_plt3_sex_zy_mz_sd_cv']

In [29]:
def mean_std(D,D_temp):
    temp=copy.deepcopy(D_temp)
    D_stats = D.describe()
    D_stats = D_stats.transpose()
    temp= (temp - D_stats['mean']) / (D_stats['std'])
    # D_num_temp[i]= (D_num_temp[i] - D_stats['min']) / (D_stats['max']-D_stats['min'])
    temp.label=D_temp.label
    return temp
# Dp=pd.read_csv('./data/py_33053.csv')
Dp=pd.read_csv('./data/py_33053_cor_hct.csv')

Dc_test=pd.read_csv('./data/cg_test_2024-10_to_2025-09_54923.csv')
Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')

/tmp/ipykernel_6702/3127532866.py:10: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  Dp=pd.read_csv('./data/py_33053_cor_hct.csv')
/tmp/ipykernel_6702/3127532866.py:13: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')


## para1

In [30]:
dauc={"lst_k":[],"cg_auc":[],"cg_size":[],"py_auc":[],"py_size":[],"method":[]}

In [31]:
def balance_under_sample(data):
    dpos=data.loc[data.label==1]
    dneg=data.loc[data.label==0]
    if dpos.shape[0]==dneg.shape[0]:
        return data
    num=min(dpos.shape[0],dneg.shape[0])
    if dpos.shape[0]>dneg.shape[0]:
        temp=dpos.sample(n=num,replace=False)
        data=pd.concat([dneg,temp],axis=0).reset_index(drop=True)
    else:
        temp=dneg.sample(n=num,replace=False)
        data=pd.concat([dpos,temp],axis=0).reset_index(drop=True)
    return data

In [33]:
time_n=0
n=1
Dp_temp=mean_std(Dc[lst[n]],Dp[lst[n]])
D_num_temp=mean_std(Dc[lst[n]],Dc_test[lst[n]])
for k in range(10):
    
    D_test=D_num_temp[lst[n]] 
    # D_test=balance_under_sample(D_test)
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    
    # model=keras.models.load_model("./models/model_para2_train329692_"+fnams[n]+'_k'+str(k)+'.keras')
    model=keras.models.load_model("./models/model_para2_drop0.15_train329692_"+fnams[n]+'_k'+str(k)+'.keras')
                   # model.save("./models/model_para2_train329692_"+fnams[n]+'_k'+str(k)+'.keras')
    prob_model = keras.Sequential([model,layers.Softmax()])
    pred=prob_model.predict(x_test)
    
    fpr = dict()
    tpr = dict()
    thresholds = dict()
    roc_auc = dict()
            # fpr,tpr,thresholds = roc_curve(test_labels, predictions,drop_intermediate=False)
    fpr,tpr,thresholds = roc_curve(y_test, pred[:,1])
    roc_auc = auc(fpr, tpr)
    
    dauc["lst_k"].append("train329692_k"+str(k))
    dauc["cg_auc"].append(roc_auc)
    dauc["cg_size"].append(D_test.shape[0])
    
    
    D_test=Dp_temp[lst[n]] 
    # D_test=balance_under_sample(D_test)
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    pred=prob_model.predict(x_test)
    fpr,tpr,thresholds = roc_curve(y_test, pred[:,1])
    roc_auc = auc(fpr, tpr)
    
    dauc["py_auc"].append(roc_auc)
    dauc["py_size"].append(D_test.shape[0])
    dauc["method"].append('dnn_drop0.15')
    # dauc["method"].append('dnn')
    # dauc["method"].append('dnn_drop0.15_train_n_test_under') 
                # 

2025-11-07 00:39:39.111721: I external/local_xla/xla/service/service.cc:163] XLA service 0x786290004ad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 00:39:39.111732: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Ti, Compute Capability 8.9
2025-11-07 00:39:39.111733: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-11-07 00:39:39.115493: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-07 00:39:39.141961: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91400
2025-11-07 00:39:39.147597: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the

 593/1717 ━━━━━━━━━━━━━━━━━━━━ 0s 254us/step

I0000 00:00:1762447180.334570    7877 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1717/1717 ━━━━━━━━━━━━━━━━━━━━ 3s 793us/step
 994/1033 ━━━━━━━━━━━━━━━━━━━━ 0s 303us/step

2025-11-07 00:39:42.306893: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:39:42.306918: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:39:43.077445: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_39', 16 bytes spill stores, 16 bytes spill loads



...

1033/1033 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


2025-11-07 00:40:43.087223: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:43.087257: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:43.087264: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:43.087269: I external/l

1717/1717 ━━━━━━━━━━━━━━━━━━━━ 3s 704us/step
 957/1033 ━━━━━━━━━━━━━━━━━━━━ 0s 315us/step

2025-11-07 00:40:46.501505: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:46.501540: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:46.501547: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-07 00:40:46.501551: I external/l

1033/1033 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [15]:
# res.to_csv("train329692_lst_group(tim_plt3)_para2_dnn.csv")

In [16]:
# model=keras.models.load_model("./models/model_para2_train329692_"+fnams[n]+'_k'+str(0)+'.keras')
# model.summary()

In [106]:
d={"features":['male', 'female','zy', 'mz', 'age', 'wbc', 'neu', 'lym', 'mon','eos', 
               'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label']}
arr=[]
for i in range(1,len(model.layers)):
    # print(i)
    if i!=11:
        temp={"weights":[],"bias":[]}
        temp["weights"]=model.layers[i].weights[0].numpy().tolist()
        temp["bias"]=model.layers[i].weights[1].numpy().tolist()
        arr.append(temp)
d["model_saa"]=arr
# Save to JSON file
import json
with open("model_saa.json", "w") as f:
    json.dump(d, f)

## xgb-lgb

In [23]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.utils import shuffle
import joblib

In [24]:
time_n=0
n=1
Dp_temp=Dp[lst[n]]
D_num_temp=Dc_test[lst[n]]
for k in range(10):
    
    D_test=D_num_temp[lst[n]]  
    D_test=balance_under_sample(D_test)
    
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    
    model = XGBClassifier()
#joblib.dump(xgb_model,"./models_xgb_lgb/tune_xgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')
    model= joblib.load("./models_xgb_lgb/tune_xgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')
    pred=model.predict_proba(x_test)
    
    fpr = dict()
    tpr = dict()
    thresholds = dict()
    roc_auc = dict()
    #         fpr,tpr,thresholds = roc_curve(test_labels, predictions,drop_intermediate=False)
    fpr,tpr,thresholds = roc_curve(y_test, pred[:,1])
    roc_auc = auc(fpr, tpr)
    
    dauc["lst_k"].append("train329692_k"+str(k))
    dauc["cg_auc"].append(roc_auc)
    dauc["cg_size"].append(D_test.shape[0])
    
    
    D_test=Dp_temp[lst[n]] 
    D_test=balance_under_sample(D_test)
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    pred=model.predict_proba(x_test)
    fpr,tpr,thresholds = roc_curve(y_test, pred[:,1])
    roc_auc = auc(fpr, tpr)
    
    dauc["py_auc"].append(roc_auc)
    dauc["py_size"].append(D_test.shape[0])
    # dauc["method"].append('tune_xgb')
    dauc["method"].append('tune_xgb_train_n_test_under')

In [3]:
time_n=0
n=1
Dp_temp=Dp[lst[n]]
D_num_temp=Dc_test[lst[n]]
for k in range(10):
    
    D_test=D_num_temp[lst[n]] 
    D_test=balance_under_sample(D_test)
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    
    model = XGBClassifier()
     #joblib.dump(bst, "./models_xgb_lgb/tune_lgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')
    model= joblib.load("./models_xgb_lgb/tune_lgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')
    pred=model.predict(x_test)
    
    fpr = dict()
    tpr = dict()
    thresholds = dict()
    roc_auc = dict()
    #         fpr,tpr,thresholds = roc_curve(test_labels, predictions,drop_intermediate=False)
    fpr,tpr,thresholds = roc_curve(y_test, pred)
    roc_auc = auc(fpr, tpr)
    
    dauc["lst_k"].append("train329692_k"+str(k))
    dauc["cg_auc"].append(roc_auc)
    dauc["cg_size"].append(D_test.shape[0])
    
    
    D_test=Dp_temp[lst[n]]  
    D_test=balance_under_sample(D_test)
    x_test,y_test=D_test.drop('label',axis=1),D_test['label']
    pred=model.predict(x_test)
    fpr,tpr,thresholds = roc_curve(y_test, pred)
    roc_auc = auc(fpr, tpr)
    
    dauc["py_auc"].append(roc_auc)
    dauc["py_size"].append(D_test.shape[0])
    # dauc["method"].append('tune_lgb')
    dauc["method"].append('tune_lgb_train_n_test_under')

NameError: name 'Dp' is not defined

In [21]:
# res.to_csv("train329692_lst_group(tim_plt3)_dnn_drop0.15_tune_xgb_lgb.csv")

In [28]:
res.to_csv("train329692_lst_group(tim_plt3)_dnn_drop0.15_tune_xgb_lgb_test_under.csv")